In [4]:
import pandas as pd
import numpy as np
import seaborn as srn
import matplotlib.pyplot as plt

In [5]:
wego = pd.read_csv("./Headway Data, 8-1-2023 to 9-30-2023.csv")

In [6]:
wego.head()

,CALENDAR_ID,SERVICE_ABBR,ADHERENCE_ID,DATE,ROUTE_ABBR,BLOCK_ABBR,OPERATOR,TRIP_ID,OVERLOAD_ID,ROUTE_DIRECTION_NAME,...,ACTUAL_HDWY,HDWY_DEV,ADJUSTED_EARLY_COUNT,ADJUSTED_LATE_COUNT,ADJUSTED_ONTIME_COUNT,STOP_CANCELLED,PREV_SCHED_STOP_CANCELLED,IS_RELIEF,DWELL_IN_MINS,SCHEDULED_LAYOVER_MINUTES
0,120230801,1,99457890,2023-08-01,22,2200,1040,345104,0,TO DOWNTOWN,...,NaN,NaN,0,0,1,0,0.0,0,6.500000,NaN
1,120230801,1,99457891,2023-08-01,22,2200,1040,345104,0,TO DOWNTOWN,...,NaN,NaN,0,0,1,0,0.0,0,0.000000,NaN
2,120230801,1,99457892,2023-08-01,22,2200,1040,345104,0,TO DOWNTOWN,...,NaN,NaN,0,0,1,0,0.0,0,0.000000,NaN
3,120230801,1,99457893,2023-08-01,22,2200,1040,345104,0,TO DOWNTOWN,...,NaN,NaN,0,0,1,0,NaN,0,0.000000,NaN
4,120230801,1,99457894,2023-08-01,22,2200,1040,345105,0,FROM DOWNTOWN,...,NaN,NaN,0,0,1,0,0.0,0,12.866666,5.0


In [7]:
wego.columns

Index(['CALENDAR_ID', 'SERVICE_ABBR', 'ADHERENCE_ID', 'DATE', 'ROUTE_ABBR',
       'BLOCK_ABBR', 'OPERATOR', 'TRIP_ID', 'OVERLOAD_ID',
       'ROUTE_DIRECTION_NAME', 'TIME_POINT_ABBR', 'ROUTE_STOP_SEQUENCE',
       'TRIP_EDGE', 'LATITUDE', 'LONGITUDE', 'SCHEDULED_TIME',
       'ACTUAL_ARRIVAL_TIME', 'ACTUAL_DEPARTURE_TIME', 'ADHERENCE',
       'SCHEDULED_HDWY', 'ACTUAL_HDWY', 'HDWY_DEV', 'ADJUSTED_EARLY_COUNT',
       'ADJUSTED_LATE_COUNT', 'ADJUSTED_ONTIME_COUNT', 'STOP_CANCELLED',
       'PREV_SCHED_STOP_CANCELLED', 'IS_RELIEF', 'DWELL_IN_MINS',
       'SCHEDULED_LAYOVER_MINUTES'],
      dtype='object')

In the data, the bus route can be identified by its ROUTE_ABBR value.
3: West End
7: Hillsboro
22: Bordeaux
23: Dickerson Pike
50: Charlotte Pike
52: Nolensville Pike
55: Murfreesboro Pike
56: Gallatin Pike

In [8]:
wego.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350329 entries, 0 to 350328
Data columns (total 30 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CALENDAR_ID                350329 non-null  int64  
 1   SERVICE_ABBR               350329 non-null  int64  
 2   ADHERENCE_ID               350329 non-null  int64  
 3   DATE                       350329 non-null  object 
 4   ROUTE_ABBR                 350329 non-null  int64  
 5   BLOCK_ABBR                 350329 non-null  int64  
 6   OPERATOR                   350329 non-null  int64  
 7   TRIP_ID                    350329 non-null  int64  
 8   OVERLOAD_ID                350329 non-null  int64  
 9   ROUTE_DIRECTION_NAME       350329 non-null  object 
 10  TIME_POINT_ABBR            350329 non-null  object 
 11  ROUTE_STOP_SEQUENCE        350318 non-null  float64
 12  TRIP_EDGE                  350329 non-null  int64  
 13  LATITUDE                   35

In [9]:
type(wego)

pandas.core.frame.DataFrame

In [14]:
wego['ROUTE_ABBR'].value_counts().sort_index()

ROUTE_ABBR
3     47162
7     18026
22    25959
23    42108
50    43291
52    51819
55    61944
56    60020
Name: count, dtype: int64

The trip can be identified by the DATE/CALENDAR_ID plus the TRIP_ID.
Warning: The TRIP_ID refers to the route and time but will be used across multiple days.

The data contains multiple time points for each trip. There are more stops along the route than time points, but the time points are the points with specific scheduled times the bus operators must adhere to.

The first stop of a trip has a TRIP_EDGE of 1, the last has a TRIP_EDGE of 2, and the intermediate stops are TRIP_EDGE 0.

Here is the first trip in the dataset. It was a Bordeaux route (Route 22), scheduled to start at 4:42:00 and end at 5:10:00.

In [15]:
wego[['DATE', 'CALENDAR_ID', 'TRIP_ID', 'ROUTE_ABBR', 'TIME_POINT_ABBR', 'TRIP_EDGE', 'SCHEDULED_TIME']].loc[:3]

,DATE,CALENDAR_ID,TRIP_ID,ROUTE_ABBR,TIME_POINT_ABBR,TRIP_EDGE,SCHEDULED_TIME
0,2023-08-01,120230801,345104,22,MHSP,1,2023-08-01 04:42:00
1,2023-08-01,120230801,345104,22,ELIZ,0,2023-08-01 04:46:00
2,2023-08-01,120230801,345104,22,CV23,0,2023-08-01 04:54:00
3,2023-08-01,120230801,345104,22,MCC5_10,2,2023-08-01 05:10:00


Note that the same TRIP_ID appears on the following day.

In [17]:
(
    wego
    .loc[wego['DATE'].astype(str).isin(['2023-08-01', '2023-08-02'])]
    .loc[wego['TRIP_ID'] == 345104]
    [['DATE', 'ROUTE_ABBR', 'TRIP_ID', 'TIME_POINT_ABBR', 'TRIP_EDGE', 'ROUTE_DIRECTION_NAME', 'SCHEDULED_TIME', 'ROUTE_STOP_SEQUENCE']]
)

,DATE,ROUTE_ABBR,TRIP_ID,TIME_POINT_ABBR,TRIP_EDGE,ROUTE_DIRECTION_NAME,SCHEDULED_TIME,ROUTE_STOP_SEQUENCE
0,2023-08-01,22,345104,MHSP,1,TO DOWNTOWN,2023-08-01 04:42:00,14.0
1,2023-08-01,22,345104,ELIZ,0,TO DOWNTOWN,2023-08-01 04:46:00,10.0
2,2023-08-01,22,345104,CV23,0,TO DOWNTOWN,2023-08-01 04:54:00,5.0
3,2023-08-01,22,345104,MCC5_10,2,TO DOWNTOWN,2023-08-01 05:10:00,1.0
6461,2023-08-02,22,345104,MHSP,1,TO DOWNTOWN,2023-08-02 04:42:00,14.0
6462,2023-08-02,22,345104,ELIZ,0,TO DOWNTOWN,2023-08-02 04:46:00,10.0
6463,2023-08-02,22,345104,CV23,0,TO DOWNTOWN,2023-08-02 04:54:00,5.0
6464,2023-08-02,22,345104,MCC5_10,2,TO DOWNTOWN,2023-08-02 05:10:00,1.0


Q1
What is the overall on-time performance, and what do the overall distribution of adherence look like?

In [10]:
# tracking if adherence column meets guidelines and storing boolean into a new column
wego['adh_track'] = (wego[['ADHERENCE']] > -6) & (wego[['ADHERENCE']] < 1)

In [11]:
# looking at the new column made to compare
wego[['adh_track', 'ADHERENCE']]

,adh_track,ADHERENCE
0,True,-2.133333
1,True,-2.450000
2,True,-0.933333
3,False,6.283333
4,True,-1.583333
...,...,...
350324,False,-8.433333
350325,False,-11.300000
350326,True,-4.316666
350327,False,-22.083333


In [12]:
# filtered out the False booleans
on_time = wego[wego['adh_track'] == True] 

# filters out the TRIP_EDGE type 2 since these are not considered early or late b/c it is the last stop of the day
not_last_stop = on_time[on_time['TRIP_EDGE'] != 2]

not_last_stop[['TRIP_EDGE']]

,TRIP_EDGE
0,1
1,0
2,0
4,1
5,0
...,...
350314,1
350315,0
350317,1
350320,1


In [13]:
#find the %age of how many are on time
overall_on_time = len(on_time) / len(wego) * 100

overall_on_time

70.28222042708425

In [18]:
# For Question 5
relate = wego.copy()
relate['LATE'] = np.where(
    (relate['ADHERENCE'].notnull()) & (relate['ADHERENCE'] <= -6),
    'Yes',
    'No'
)
relate[['LATE', 'ADHERENCE']]
# workingtable[['ON_TIME', 'ADHERENCE']]

,LATE,ADHERENCE
0,No,-2.133333
1,No,-2.450000
2,No,-0.933333
3,No,6.283333
4,No,-1.583333
...,...,...
350324,Yes,-8.433333
350325,Yes,-11.300000
350326,No,-4.316666
350327,Yes,-22.083333


Q3. How does time of day or day of week affect on-time performance?

Notes: Sometimes, weekend service runs on a weekday, e.g. during holidays, so need to account for this. 

In [21]:
wego['SCHEDULED_TIME'].describe()

count                  350329
unique                  70792
top       2023-08-14 17:05:00
freq                       22
Name: SCHEDULED_TIME, dtype: object

In [22]:
wego['CALENDAR_ID'].describe()

count    3.503290e+05
mean     1.202309e+08
std      5.063417e+01
min      1.202308e+08
25%      1.202308e+08
50%      1.202308e+08
75%      1.202309e+08
max      1.202309e+08
Name: CALENDAR_ID, dtype: float64

In [26]:
wego['SERVICE_ABBR'].describe()

count    350329.000000
mean          1.298465
std           0.633101
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           3.000000
Name: SERVICE_ABBR, dtype: float64

In [ ]:
pd.crosstab(wego['SCHEDULED_TIME'], wego['ADHERENCE']).plot(kind = 'bar', 
                                                          stacked = False,       
                                                          color = ['cornflowerblue', 'coral', 'pink'],     
                                                          edgecolor = 'black')              
plt.title('Adherence by Scheduled Time')                   
plt.xticks(rotation = 0)
plt.show()

In [ ]:
wego.groupby('SCHEDULED_TIME', ['ADHERENCE'].describe

In [ ]:
plt.figure(figsize = (10,6))

sns.scatterplot(data = wego,
               x = 'SCHEDULED_TIME',
               y = 'ACTUAL_ARRIVAL_TIME',
               hue = 'ADHERENCE',
               palette = ['cornflowerblue', 'coral', 'pink'])
plt.show()

In [ ]:
wego.'SCHEDULED TIME'.value_counts()

In [ ]:
plt.plot(wego.SCHEDULED_TIME, wego.ADHERENCE)
plt.show()